<div align="center">

<a href="https://colab.research.google.com/github/utkukose/machine-ethics-ai-safety-NB-lecture/blob/main/exams/final/FIN1_clinical_safety_case.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Final Capstone: Building and Auditing a Safer AI System

## Project 1: A safety case for a clinical risk model

**Machine Ethics and Artificial Intelligence Safety (11118BLG001)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## The project

A malignancy classifier trained on a public diagnostic dataset is examined for adversarial robustness, calibration, conformal coverage and detection of distribution shift, and the evidence is assembled into claims, arguments and evidence [1, 2, 3, 4].

**Required extensions.** Add a non-linear model and a stronger attack, such as projected gradient descent [5], report class-conditional conformal coverage, and evaluate at least two realistic shift scenarios with a monitor that raises alarms. Rewrite the safety case in the Goal Structuring Notation, identify the weakest claim and state what additional evidence would be needed before any clinical use.

## What this example implements

The notebook trains a logistic model, measures accuracy under FGSM attacks with and without adversarial training, calibrates probabilities with temperature scaling, builds 95 percent conformal prediction sets, simulates device drift with a Mahalanobis monitor and assembles a claims table.

Run the cells in order. The example is a baseline: the capstone extends it, evaluates the extensions and reports the results.

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. Robustness

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize_scalar

data = load_breast_cancer()
X, y = data.data, 1 - data.target            # 1 = malignant
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.4, random_state=SEED, stratify=y)
X_cal, X_te, y_cal, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=SEED, stratify=y_tmp)
sc = StandardScaler().fit(X_tr)
Z_tr, Z_cal, Z_te = sc.transform(X_tr), sc.transform(X_cal), sc.transform(X_te)
model = LogisticRegression(C=0.5, max_iter=5000).fit(Z_tr, y_tr)
w, b = model.coef_[0], model.intercept_[0]

def acc(Z, yv, w=w, b=b):
    return float(np.mean(((Z @ w + b) > 0).astype(int) == yv))

def fgsm(Z, yv, eps, w=w, b=b):
    p = 1 / (1 + np.exp(-(Z @ w + b)))
    return Z + eps * np.sign((p - yv)[:, None] * w[None, :])

eps_grid = np.linspace(0, 0.5, 11)
robust = [acc(fgsm(Z_te, y_te, e), y_te) for e in eps_grid]
wa, ba = w.copy(), b
for _ in range(3):
    Z_aug, y_aug = np.r_[Z_tr, fgsm(Z_tr, y_tr, 0.2, wa, ba)], np.r_[y_tr, y_tr]
    m2 = LogisticRegression(C=0.5, max_iter=5000).fit(Z_aug, y_aug)
    wa, ba = m2.coef_[0], m2.intercept_[0]
robust_at = [acc(fgsm(Z_te, y_te, e, wa, ba), y_te, wa, ba) for e in eps_grid]
margin = np.abs(Z_te @ w + b) / np.abs(w).sum()
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(eps_grid, robust, "o-", label="standard training"); ax[0].plot(eps_grid, robust_at, "s-", label="adversarial training")
ax[0].set_xlabel("FGSM budget (standardised units)"); ax[0].set_ylabel("accuracy"); ax[0].legend()
ax[1].hist(margin, bins=30); ax[1].set_xlabel("smallest L-infinity change that flips the decision"); plt.tight_layout(); plt.show()

## 2. Calibration, conformal prediction and shift

In [ ]:
def ece(p, yv, bins=10):
    edges, e = np.linspace(0, 1, bins + 1), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (p >= lo) & ((p < hi) if hi < 1 else (p <= hi))
        if m.any():
            e += m.mean() * abs(p[m].mean() - yv[m].mean())
    return e

sig = lambda t: 1 / (1 + np.exp(-t))
logit_cal, logit_te = Z_cal @ w + b, Z_te @ w + b
nll = lambda T: -np.mean(y_cal * np.log(sig(logit_cal / T) + 1e-12) + (1 - y_cal) * np.log(1 - sig(logit_cal / T) + 1e-12))
T = minimize_scalar(nll, bounds=(0.05, 20), method="bounded").x
p_te, p_te_T = sig(logit_te), sig(logit_te / T)
p_cal = sig(logit_cal / T)
scores = 1 - np.where(y_cal == 1, p_cal, 1 - p_cal)
alpha = 0.05
qhat = np.quantile(scores, np.ceil((len(scores) + 1) * (1 - alpha)) / len(scores), method="higher")

def pred_sets(p):
    return [[k for k, pk in ((0, 1 - pi), (1, pi)) if 1 - pk <= qhat] for pi in p]

sets = pred_sets(p_te_T)
coverage = np.mean([yv in s for yv, s in zip(y_te, sets)])
sizes = np.array([len(s) for s in sets])
shift = np.zeros(Z_te.shape[1]); shift[:5] = 0.8          # drift in five features, for example after a device change
Z_shift = Z_te + shift
cov_shift = np.mean([yv in s for yv, s in zip(y_te, pred_sets(sig((Z_shift @ w + b) / T)))])
mu = Z_tr.mean(0)
icov = np.linalg.inv(np.cov(Z_tr, rowvar=False) + 1e-3 * np.eye(Z_tr.shape[1]))
maha = lambda Z: np.einsum("ij,jk,ik->i", Z - mu, icov, Z - mu)
auroc = roc_auc_score(np.r_[np.zeros(len(Z_te)), np.ones(len(Z_shift))], np.r_[maha(Z_te), maha(Z_shift)])
print(f"ECE {ece(p_te, y_te):.3f} before and {ece(p_te_T, y_te):.3f} after temperature scaling (T = {T:.2f})")
print(f"conformal coverage {coverage:.3f}, mean set size {sizes.mean():.2f}; coverage under shift {cov_shift:.3f}; drift monitor AUROC {auroc:.3f}")

## 3. A first safety case

In [ ]:
case = pd.DataFrame([
    ("C1 The model separates malignant from benign cases", "Held-out evaluation", f"accuracy {acc(Z_te, y_te):.3f}, AUROC {roc_auc_score(y_te, p_te):.3f}"),
    ("C2 Stated confidence is calibrated", "Reliability analysis with temperature scaling", f"ECE {ece(p_te, y_te):.3f} before, {ece(p_te_T, y_te):.3f} after"),
    ("C3 Uncertainty carries a coverage guarantee", "Split conformal prediction at 95 percent", f"coverage {coverage:.3f}, mean set size {sizes.mean():.2f}"),
    ("C4 Small perturbations do not flip decisions", "FGSM evaluation and adversarial training", f"accuracy at budget 0.2: {robust[4]:.3f} standard, {robust_at[4]:.3f} adversarial"),
    ("C5 Distribution shift is detected", "Mahalanobis monitor on simulated drift", f"AUROC {auroc:.3f}; coverage under shift {cov_shift:.3f}"),
], columns=["Claim", "Argument", "Evidence"])
case

## Report and submission

The technical report states the question, explains the design choices, presents the experiments with the figures produced by the notebook and discusses the ethical and safety implications and the limitations of the results. It cites at least eight sources in square brackets, at least four of them from the course readings, and links to the final Colab notebook.

This capstone also supports self-learning and can be completed at any pace. When the course is taught actively in a semester, the final capstone is sent by e-mail to utkukose@sdu.edu.tr or utkukose@gmail.com no later than 23:53 (Türkiye time) on the last Sunday of Week 14, with the report and all code files attached or linked.

## References

[1] Goodfellow, I. J., Shlens, J., & Szegedy, C. (2015). Explaining and harnessing adversarial examples. In *International Conference on Learning Representations (ICLR)*. <https://arxiv.org/abs/1412.6572>

[2] Guo, C., Pleiss, G., Sun, Y., & Weinberger, K. Q. (2017). On calibration of modern neural networks. In *Proceedings of the 34th International Conference on Machine Learning, PMLR 70* (pp. 1321-1330).

[3] Angelopoulos, A. N., & Bates, S. (2023). Conformal prediction: A gentle introduction. *Foundations and Trends in Machine Learning*, 16(4), 494-591. <https://doi.org/10.1561/2200000101>

[4] Kelly, T., & Weaver, R. (2004). The Goal Structuring Notation: A safety argument notation. In *Proceedings of the Dependable Systems and Networks 2004 Workshop on Assurance Cases*.

[5] Madry, A., Makelov, A., Schmidt, L., Tsipras, D., & Vladu, A. (2018). Towards deep learning models resistant to adversarial attacks. In *International Conference on Learning Representations (ICLR)*. <https://arxiv.org/abs/1706.06083>